# Feature ablation: what signal remains beyond command vocabulary?

This notebook reproduces the CPU experiment in `scripts/run_feature_ablation.py`. The corpus is the pinned CC BY 4.0 Cyber Task Horizons release (177 non-empty CTF traces, 40 benchmark-qualified tasks). Human terminal recordings and Agent tool-call logs use different capture paths, so all classification results are source-label separability diagnostics, not evidence of identifying an unknown decision maker.

The outer split is grouped by benchmark and task. F3−F1 is evaluated on matched tasks; prefix comparisons use the same eligible cohort. Raw logs are read from the local ignored `data/` directory and are never copied to results.

In [ ]:
from pathlib import Path
import json, subprocess, sys
ROOT = Path.cwd()
if not (ROOT / 'scripts' / 'run_feature_ablation.py').exists(): ROOT = ROOT.parent
assert (ROOT / 'scripts' / 'run_feature_ablation.py').exists()
PYTHON = ROOT / '.venv' / 'bin' / 'python'
print('Repository:', ROOT)
print('Python:', subprocess.check_output([str(PYTHON), '--version'], text=True).strip())

## Recompute

The runner records the six predeclared task splits, matched observation cohorts, five order permutations, task-level metrics, and predictions. It uses no external API or GPU.

In [ ]:
run = subprocess.run([str(PYTHON), 'scripts/run_feature_ablation.py'], cwd=ROOT, check=True, text=True, capture_output=True)
print(run.stdout[-5000:])

In [ ]:
import pandas as pd
from IPython.display import display, Image
result = json.loads((ROOT / 'results' / 'feature_ablation_v1.json').read_text())
rows = []
for item in result['primary_summary']:
    rows.append({'features': item['feature'], 'classifier': item['classifier'],
                 'balanced accuracy': item['metrics']['balanced_accuracy'],
                 '95% task CI': item['ci95_task_cluster']['balanced_accuracy'],
                 'macro F1': item['metrics']['macro_f1'], 'AUROC': item['metrics']['auroc'],
                 'Human recall': item['metrics']['human_recall'], 'Agent recall': item['metrics']['agent_recall']})
display(pd.DataFrame(rows).round(3))
print(result['interpretation_guard'])

In [ ]:
display(pd.DataFrame(result['diagnostic_summary']).round(3))
display(Image(filename=str(ROOT / 'results' / 'figures' / 'feature_ablation_v1.png')))

In [ ]:
# Derive paired contrasts, transfer diagnostics, and publication figures from the same task-level outputs.
for command in (
    ['scripts/summarize_feature_contrasts.py'],
    ['scripts/run_agent_model_transfer.py', '--bootstraps', '5000'],
    ['scripts/plot_feature_ablation.py'],
): 
    step = subprocess.run([str(PYTHON), *command], cwd=ROOT, check=True, text=True, capture_output=True)
    print(command[0], step.stdout[-1200:])
contrast = json.loads((ROOT / 'results' / 'feature_ablation_contrasts_v1.json').read_text())
transfer = json.loads((ROOT / 'results' / 'agent_model_transfer_v1.json').read_text())
print('Matched feature, budget, and order contrasts:')
display(contrast['contrasts'])
display(pd.DataFrame([{'held-out model': x['heldout_model'], 'feature': x['feature'], 'BA': x['metrics']['balanced_accuracy'], '95% task CI': x['ci95_task_cluster']['balanced_accuracy']} for x in transfer['summary'] if x['feature'] in ('F1_command_tfidf','F3_command_plus_structure')]).round(3))
display(Image(filename=str(ROOT / 'results' / 'figures' / 'feature_budget_order_v1.png')))

## Interpretation boundary

A positive F3−F1 result would show incremental source-label signal in this release under task holdout. It would not isolate execution mode from capture tooling. A negative result would bound the tested feature set and cohort only. CTF same-task labels are not attack-activity labels.